In [12]:
import os
import uuid
from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding
from dotenv import load_dotenv

load_dotenv()



True

In [13]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION_NAME = "financial"
FILE_PATH = "./AAPL_10-K_1A_temp.md"

qdrant = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
)


In [8]:
qdrant.delete_collection(COLLECTION_NAME)

True

In [9]:

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=384,
        distance=models.Distance.COSINE,
    ),
)

True

In [14]:
from markdown_it.rules_block.paragraph import paragraph
with open(FILE_PATH, "r", encoding="utf-8") as f:
    content = f.read()
    
paragraphs = content.split("\n\n")
chunks = [p.strip() for p in paragraphs if len(p.strip()) > 50]

chunks[0]



'The Company’s business, reputation, results of operations, financial condition and stock price can be affected by a number of factors, whether currently known or unknown, including those described below. When any one or more of these risks materialize from time to time, the Company’s business, reputation, results of operations, financial condition and stock price can be materially and adversely affected.'

In [15]:
model = TextEmbedding(MODEL_NAME)

points = []
for chunk in chunks:
    embedding = list(model.passage_embed([chunk]))[0].tolist()
    point = models.PointStruct(
        id=str(uuid.uuid4()),
        vector=embedding,
        payload={"text": chunk, "source": FILE_PATH},
    )
    points.append(point)

qdrant.upload_points(collection_name=COLLECTION_NAME, points=points)


In [21]:
query_text = "What are the main financial risks?"
query_embedding = list(model.query_embed([query_text]))[0].tolist()

results = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    query=query_embedding,
    limit=3,
)

In [23]:
for r in results.points:
    print(f"Score: {r.score}")
    print(f"Texto: {r.payload['text'][:100]}...")
    print("-" * 80)

Score: 0.6003566
Texto: The Company’s business, reputation, results of operations, financial condition and stock price can b...
--------------------------------------------------------------------------------
Score: 0.6003566
Texto: The Company’s business, reputation, results of operations, financial condition and stock price can b...
--------------------------------------------------------------------------------
Score: 0.5712656
Texto: Adverse economic conditions can also lead to increased credit and collectibility risk on the Company...
--------------------------------------------------------------------------------
